In [1]:
import numpy as np
import pandas as pd
from types import resolve_bases
import pickle
import plotly.express as px
from SamplingMethods import Sampler_class
from ax.api.client import Client
from ax.api.configs import RangeParameterConfig
from ax.generation_strategy.center_generation_node import CenterGenerationNode
from ax.generation_strategy.transition_criterion import MinTrials
from ax.generation_strategy.generation_strategy import GenerationStrategy
from ax.generation_strategy.generation_node import GenerationNode
from ax.generation_strategy.model_spec import GeneratorSpec
from ax.modelbridge.registry import Generators
from gpytorch.kernels import MaternKernel
from botorch.models import SingleTaskGP
from botorch.models.transforms.input import Warp
from botorch.models.map_saas import AdditiveMapSaasSingleTaskGP
from ax.utils.stats.model_fit_stats import MSE
from ax.models.torch.botorch_modular.surrogate import SurrogateSpec, ModelConfig
from botorch.acquisition.logei import qLogNoisyExpectedImprovement

In [2]:
client = Client()
gp_model = client.load_from_json_file("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/StoichModelGP/ModelGP_M25.json")
gp_model.get_next_trials(max_trials=1)

{56: {'n_ci': 0.7708670939735744, 'n_it': 0.3980077276838163}}

In [3]:
def SurrogateModelOfReality(n_ci,n_it):
    y_pred = gp_model.predict([{"n_ci":n_ci,"n_it":n_it}])[0]["t1"][0]
    return np.float64(y_pred)

In [4]:
class OptimisationSetup_class(object):
    def __init__(self):
        self.Parameters_lis = [
            RangeParameterConfig(name="s1", parameter_type="float", bounds=(0, 1)),
            RangeParameterConfig(name="s2", parameter_type="float", bounds=(0, 1)),
            RangeParameterConfig(name="b1", parameter_type="float", bounds=(0, 1)),
        ]
OptimisationSetup_obj = OptimisationSetup_class()

In [5]:
def PredictorsToCaStoichs(s1,b1):
    return (0.5+(2.0-0.5)*s1)*b1

def PredictorsToIaStoichs(s2,b1):
    return (1.0+(2.0-1.0)*s2)*(1-b1)

In [6]:
y_max_lis = []

for i in range(100):
    client = Client()
    parameters = [
        RangeParameterConfig(
            name="s1", parameter_type="float", bounds=(0, 1)
        ),
        RangeParameterConfig(
            name="s2", parameter_type="float", bounds=(0, 1)
        ),
        RangeParameterConfig(
            name="b1", parameter_type="float", bounds=(0, 1)
        ),
    ]
    client.configure_experiment(parameters=parameters)
    def construct_generation_strategy(
        generator_spec: GeneratorSpec, node_name: str,
    ) -> GenerationStrategy:
        """Constructs a Center + Sobol + Modular BoTorch `GenerationStrategy`
        using the provided `generator_spec` for the Modular BoTorch node.
        """
        botorch_node = GenerationNode(
            node_name=node_name,
            model_specs=[generator_spec],
        )
        return GenerationStrategy(
            name=f"{node_name}",
            nodes=[botorch_node]
        )

    # Let's construct the simplest version with all defaults.
    construct_generation_strategy(
        generator_spec=GeneratorSpec(model_enum=Generators.BOTORCH_MODULAR),
        node_name="Modular BoTorch",
    )

    surrogate_spec = SurrogateSpec(
        model_configs=[
            # Select between two models:
            # An additive mixture of relatively strong SAAS priors with input Warping.
            # A relatively vanilla GP with a Matern kernel.
            ModelConfig(
                botorch_model_class=SingleTaskGP,
                covar_module_class=MaternKernel,
                covar_module_options={"nu": 2.5},
            ),
        ],
        eval_criterion=MSE,  # Select the model to use as the one that minimizes mean squared error.
        allow_batched_models=False,  # Forces each metric to be modeled with an independent BoTorch model.
        # If we wanted to specify different options for different metrics.
        # metric_to_model_configs: dict[str, list[ModelConfig]]
    )

    generator_spec = GeneratorSpec(
        model_enum=Generators.BOTORCH_MODULAR,
        model_kwargs={
            "surrogate_spec": surrogate_spec,
            "botorch_acqf_class": qLogNoisyExpectedImprovement,
            # Can be used for additional inputs that are not constructed
            # by default in Ax. We will demonstrate below.
            "acquisition_options": {},
        },
        # We can specify various options for the optimizer here.
        model_gen_kwargs = {
            "model_gen_options": {
                "optimizer_kwargs": {
                    "num_restarts": 20,
                    "sequential": False,
                    "options": {
                        "batch_limit": 5,
                        "maxiter": 200,
                    },
                },
            },
        }
    )

    generation_strategy = construct_generation_strategy(
        generator_spec=generator_spec,
        node_name="BoTorch w/ Model Selection",
    )
    generation_strategy

    client.set_generation_strategy(
        generation_strategy=generation_strategy,
    )

    metric_name = "t1" # this name is used during the optimization loop in Step 5
    objective = f"{metric_name}" # minimization is specified by the negative sign

    client.configure_optimization(objective=objective)

    # Quasirandom Sampling Exercise
    sampler_obj = Sampler_class()
    Parameters_lis = [
        {"name":"s1", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"s2", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"b1", "type":"range","bounds":[0,1],"value_type":"float"}
    ]
    X = sampler_obj.three.QuasirandomSampler3D_func(8,Parameters_lis).T

    for array in X:
        s1 = array[0]
        s2 = array[1]
        b1 = array[2]
        n_ci = PredictorsToCaStoichs(s1,b1)
        n_it = PredictorsToIaStoichs(s2,b1)
        my_parameters = {"s1": s1, "s2": s2, "b1": b1}
        trial_index = client.attach_trial(parameters=my_parameters)
        client.complete_trial(trial_index=trial_index,raw_data={"t1": SurrogateModelOfReality(n_ci,n_it)})

    for _ in range(21): # Run 21 rounds of trials
        trial = sampler_obj.McIntersiteProjTh.McIntersiteProjTh_func(OptimisationSetup_obj,client)
        s1 = trial[0][0]
        s2 = trial[0][1]
        b1 = trial[0][2]
        parameters = {"s1":s1,"s2":s2,"b1":b1}
        trial_index = client.attach_trial(parameters=parameters)
        n_ci = PredictorsToCaStoichs(s1,b1)
        n_it = PredictorsToIaStoichs(s2,b1)
        result = SurrogateModelOfReality(n_ci,n_it)
        raw_data = {metric_name: result}
        client.complete_trial(trial_index=trial_index, raw_data=raw_data)
    client._experiment.trials.pop(28)
    client._experiment.trials.pop(27)
    print(f"Trial {i} =========================================")
    y_max = np.max(np.array(client.summarize().t1))
    print(y_max)
    y_max_lis.append(y_max)
    print()

y_max_arr = np.array(y_max_lis)
print(y_max_arr)

Trial 0 =========================================
13.445383469470581

Trial 1 =========================================
13.426468157688806

Trial 2 =========================================
15.893453969691638

Trial 3 =========================================
16.633547244443914

Trial 4 =========================================
14.153781633031473

Trial 5 =========================================
17.980670447731402

Trial 6 =========================================
13.751075930499063

Trial 7 =========================================
13.899430740074871

Trial 8 =========================================
13.80458565277175

Trial 9 =========================================
14.415632969198334

Trial 10 =========================================
13.240758745581278

Trial 11 =========================================
13.723466304080427

Trial 12 =========================================
14.616192644750987

Trial 13 =========================================
16.39217282605629

Trial 14 =========

In [7]:
print(f"Max = {np.max(y_max_arr)}")
print(f"Avg = {np.average(y_max_arr)}")
print(f"Std = {np.std(y_max_arr)}")

Max = 18.204255119082788
Avg = 14.709210874844187
Std = 1.4061838763126782


In [8]:
print(y_max_arr.tolist())

[13.445383469470581, 13.426468157688806, 15.893453969691638, 16.633547244443914, 14.153781633031473, 17.980670447731402, 13.751075930499063, 13.899430740074871, 13.80458565277175, 14.415632969198334, 13.240758745581278, 13.723466304080427, 14.616192644750987, 16.39217282605629, 13.277869216634056, 13.655483339243125, 14.00651762685137, 13.73270455996316, 13.747011592301032, 13.526583048772157, 16.026243340559564, 13.612869929472872, 13.981094321475569, 15.848684142054614, 13.498707899311647, 15.4862912495627, 13.838191619431656, 16.973277270033325, 13.74661984598209, 17.772833688919572, 18.12665955861932, 13.656024440149773, 13.714756288295332, 14.129035118676544, 18.029731066587267, 16.31836525793387, 14.403309904331314, 13.802922702756044, 14.4936310096587, 13.503680295752941, 13.947321255036494, 18.204255119082788, 13.67009066512879, 13.71217815466034, 13.881053542153017, 13.813733310154745, 13.739318638168141, 14.828547584765854, 14.42032527199549, 13.769145720723293, 13.8643878041

In [9]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/TestPredModelGP_M25/DataGenerated/normal_MIPT_9_27_3.pkl"
loadeddf = pd.read_pickle(filepath_or_buffer=filepath)
latestdf = pd.DataFrame(y_max_arr)
newdf = pd.concat(objs=[loadeddf,latestdf],axis=0)
newdf = newdf.reset_index(drop=True)
pd.to_pickle(obj=newdf,filepath_or_buffer=filepath)

In [10]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/TestPredModelGP_M25/DataGenerated/normal_MIPT_9_27_3.pkl"
loadeddf = pd.read_pickle(filepath_or_buffer=filepath)
print(loadeddf)
# newdf = loadeddf.drop(loadeddf.index, inplace=True)
# pd.to_pickle(obj=newdf,filepath_or_buffer=filepath)
# print(newdf)

             0
0    13.793164
1    13.388650
2    14.744304
3    16.252553
4    14.868842
..         ...
295  15.685369
296  13.372463
297  13.819180
298  14.173143
299  13.619349

[300 rows x 1 columns]


In [11]:
# # Sanity check to make sure the MIPT is running correctly.
# df = client.summarize()
# types_lis = []
# for i in range(len(df)):
#     if i < 8:
#         types_lis.append("one-shot")
#     else:
#         types_lis.append("sequential")
# df["type"] = types_lis
# fig = px.scatter_3d(df, x='s1', y='s2', z='b1', color='type',width=1300, height=600)
# fig.show()